### import Libs :

In [93]:
from langchain.document_loaders import CSVLoader
from langchain.embeddings import HuggingFaceEmbeddings
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import joblib 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import os
import numpy as np

### Load Data

In [94]:
df = pd.read_csv('../data/questions.csv')
df.head()

,Question,Answer Status
0,Does the PDF explain what is the largest ocean...,Not Exist
1,How many moons does Jupiter have?,Not Exist
2,Tell me what is the plot of the movie 'incepti...,Not Exist
3,How do you tie a tie?,Not Exist
4,Is there information about who is the lead sin...,Not Exist


### Preprocessing & Embedding

In [ ]:
# Encode labels
le = LabelEncoder()
df['Answer Status'] = le.fit_transform(df['Answer Status'])


# Initialize BGE-M3 Embeddings
model_name = "BAAI/bge-m3"
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

def get_embeddings(questions):
    return embeddings.embed_documents(questions.tolist())

print("Generating embeddings...")
X_embeddings = np.array(get_embeddings(df['Question']))

Generating embeddings...


In [96]:
df

,Question,Answer Status
0,Does the PDF explain what is the largest ocean...,1
1,How many moons does Jupiter have?,1
2,Tell me what is the plot of the movie 'incepti...,1
3,How do you tie a tie?,1
4,Is there information about who is the lead sin...,1
...,...,...
995,What is the 'Xbox Game Bar' mentioned in Chapt...,0
996,Does the PDF explain who is the ceo of tesla i...,1
997,Does the PDF explain what is the weight of an ...,1
998,Does the book mention what is 'screencasting'?,0


### Train Model

In [97]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X_embeddings, df['Answer Status'], test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

### Evaluation

In [98]:
y_pred = clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 1.0
              precision    recall  f1-score   support

       Exist       1.00      1.00      1.00       100
   Not Exist       1.00      1.00      1.00       100

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



### Save Model

In [99]:
joblib.dump(clf, '../Models/answer_status_model_v2.pkl')
print("Model saved as answer_status_model_v2.pkl")

Model saved as answer_status_model_v2.pkl


### Test model :

In [100]:
from sentence_transformers import SentenceTransformer
import joblib

model = joblib.load('../Models/answer_status_model_v2.pkl')

def predict_question_status(question_text):
    """
    Predicts the status for a single question text using embeddings.
    """
    
    # Changed aembed_query to embed_query to avoid returning a coroutine
    v = embeddings.embed_query(question_text)
    
    # Predict using the classifier
    prediction = model.predict([v])
    return prediction[0]



In [101]:
# Example usage
question_input = "According to Chapter 9, what is the keyboard shortcut to reset the NVRAM on a 2021 MacBook Pro with an M1 chip?"
status = predict_question_status(question_input)

print(f"Question: {question_input}")
print(f"Predicted Status: {status}")

Question: According to Chapter 9, what is the keyboard shortcut to reset the NVRAM on a 2021 MacBook Pro with an M1 chip?
Predicted Status: 0
